# Preparar un embedding autocontenido para Databricks offline

Este notebook prepara y valida `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` como una sola carpeta transportable. Ejecútelo únicamente con Python 3.11 dentro de `.venv`.

No haga `git add`, commit ni push desde este notebook.

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
MODEL_SOURCE = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'
MODEL_PATH = REPO_ROOT / 'models' / 'embeddings' / MODEL_NAME

print(f'Repositorio: {REPO_ROOT}')
print(f'Python: {sys.version}')
if sys.version_info[:2] != (3, 11):
    raise RuntimeError('Este notebook requiere Python 3.11.x. Cree y seleccione .venv con py -3.11.')


Repositorio: C:\Users\tarug\Desktop\Databricks Offline
Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


## Instalar dependencias de staging

Ejecute esta celda solo después de crear y activar `.venv` con Python 3.11.

In [2]:
import subprocess

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements-staging.txt')], check=True)

CompletedProcess(args=['C:\\Users\\tarug\\AppData\\Local\\Microsoft\\WindowsApps\\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\\python.exe', '-m', 'pip', 'install', '-r', 'C:\\Users\\tarug\\Desktop\\Databricks Offline\\requirements-staging.txt'], returncode=0)

## Descargar una sola vez

El script rechaza una carpeta de destino existente para impedir sobreescrituras.

In [ ]:
import subprocess

if MODEL_PATH.exists():
    raise FileExistsError(f'El modelo ya existe y no se sobrescribirá: {MODEL_PATH}')

subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / 'scripts' / 'download_embedding.py'),
        '--model', MODEL_SOURCE,
        '--name', MODEL_NAME,
    ],
    cwd=REPO_ROOT,
    check=True,
)

In [ ]:
import json

metadata_path = MODEL_PATH / 'model-metadata.json'
metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
required = {'name', 'model_type', 'source', 'revision', 'framework', 'python_target', 'downloaded_at_utc'}
missing = required - metadata.keys()
if missing:
    raise ValueError(f'Metadata incompleta: {sorted(missing)}')
metadata

In [ ]:
files = sorted(path.relative_to(MODEL_PATH).as_posix() for path in MODEL_PATH.rglob('*') if path.is_file())
print(f'Archivos: {len(files)}')
for file_name in files:
    print(file_name)

## Validar carga offline desde la carpeta original

In [ ]:
subprocess.run(
    [sys.executable, str(REPO_ROOT / 'scripts' / 'validate_offline.py'), '--type', 'embedding', '--path', str(MODEL_PATH)],
    cwd=REPO_ROOT,
    check=True,
)

In [ ]:
import os
import numpy as np
from sentence_transformers import SentenceTransformer

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
model = SentenceTransformer(str(MODEL_PATH), local_files_only=True)
texts = [
    'Cliente solicita financiamiento para capital de trabajo.',
    'La empresa presenta crecimiento sostenido de ventas.',
]
embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
if not np.isfinite(embeddings).all():
    raise ValueError('La inferencia produjo valores no finitos.')
print(f'cantidad de textos: {len(texts)}')
print(f'shape: {embeddings.shape}')
print(f'dimensión del embedding: {embeddings.shape[1]}')
print(f'dtype: {embeddings.dtype}')

## Prueba de transportabilidad

Copia solamente la carpeta del modelo a un directorio temporal y ejecuta una inferencia offline desde esa copia.

In [ ]:
import shutil
import tempfile

with tempfile.TemporaryDirectory(prefix='offline-model-') as temp_dir:
    copied_model = Path(temp_dir) / MODEL_NAME
    shutil.copytree(MODEL_PATH, copied_model)
    copied = SentenceTransformer(str(copied_model), local_files_only=True)
    isolated_embeddings = copied.encode(['Prueba aislada.'], normalize_embeddings=True, show_progress_bar=False)
    if not np.isfinite(isolated_embeddings).all():
        raise ValueError('La prueba aislada produjo valores no finitos.')
    print(f'Prueba aislada correcta: shape={isolated_embeddings.shape}')

print('La copia temporal fue eliminada.')

## Manifest, Git LFS y tamaño

No ejecuta `git add`, commit ni push.

In [ ]:
subprocess.run([sys.executable, str(REPO_ROOT / 'scripts' / 'generate_manifest.py')], cwd=REPO_ROOT, check=True)
subprocess.run([sys.executable, str(REPO_ROOT / 'scripts' / 'verify_manifest.py')], cwd=REPO_ROOT, check=True)
subprocess.run(
    [sys.executable, str(REPO_ROOT / 'scripts' / 'verify_manifest.py'), '--model-path', str(MODEL_PATH)],
    cwd=REPO_ROOT,
    check=True,
)

In [ ]:
model_files = [path for path in MODEL_PATH.rglob('*') if path.is_file()]
largest = max(model_files, key=lambda path: path.stat().st_size)
print(f'cantidad total de archivos: {len(model_files)}')
print(f'tamaño total de carpeta: {sum(path.stat().st_size for path in model_files):,} bytes')
print(f'archivo más grande: {largest.relative_to(MODEL_PATH).as_posix()}')
print(f'tamaño archivo más grande: {largest.stat().st_size:,} bytes')

subprocess.run(['git', 'lfs', 'track'], cwd=REPO_ROOT, check=True)
weight_files = [path for path in model_files if path.suffix in {'.safetensors', '.bin', '.pt', '.pth', '.onnx', '.gguf', '.h5'}]
if not weight_files:
    raise FileNotFoundError('No se encontró un archivo de pesos compatible con Git LFS.')
subprocess.run(['git', 'check-attr', 'filter', '--', str(weight_files[0].relative_to(REPO_ROOT))], cwd=REPO_ROOT, check=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'compileall', 'scripts', 'examples'], cwd=REPO_ROOT, check=True)
subprocess.run([sys.executable, str(REPO_ROOT / 'scripts' / 'list_models.py')], cwd=REPO_ROOT, check=True)
subprocess.run(['git', 'status', '--short', '--branch'], cwd=REPO_ROOT, check=True)